In [7]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
import numpy as np
import datetime
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [8]:
# Se carga base de datos
region_0 = pd.read_csv('/datasets/geo_data_0.csv')
region_1 = pd.read_csv('/datasets/geo_data_1.csv')
region_2 = pd.read_csv('/datasets/geo_data_2.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/geo_data_0.csv'

In [ ]:
def analizar_region(datos, nombre_region):
    """
    Analiza una región: entrena modelo y calcula métricas
    """
    print(f"\n=== ANÁLISIS DE {nombre_region.upper()} ===")
    
    # Preparar datos (se elimina la columna id de features ya que es object y no se puede usar en un modelo de regresion lineal)
    features = datos.drop(['product','id'], axis=1)
    target = datos['product']
    
    # Dividir datos
    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345)
    
    # Entrenar modelo
    model = LinearRegression()
    model.fit(features_train, target_train)
    
    # Hacer predicciones
    predictions_valid = model.predict(features_valid)
    
    # Calcular métricas
    rmse = mean_squared_error(target_valid, predictions_valid)**0.5
    volumen_medio_real = target_valid.mean()
    volumen_medio_predicho = predictions_valid.mean()
    
    # Mostrar resultados
    print(f"RECM: {rmse:.2f}")
    print(f"Volumen medio real: {volumen_medio_real:.2f}")
    print(f"Volumen medio predicho: {volumen_medio_predicho:.2f}")
    print(f"Error relativo: {(rmse/volumen_medio_real)*100:.1f}%")
    
    return {
          'modelo': model,
        'rmse': rmse,
        'volumen_medio_real': volumen_medio_real,
        'volumen_medio_predicho': volumen_medio_predicho,
        'features': features,
        'target': target
    }

In [ ]:
#Se analiza las tres regiones
resultados_region_0 = analizar_region(region_0, "Region_0")
resultados_region_1 = analizar_region(region_1, "Region_1") 
resultados_region_2 = analizar_region(region_2, "Region_2")



Observaciones: El modelo de la Region_1 cuenta con el tiene el mejor rendimiento del modelo (RMSE: 0.89, error relativo: 1.3%), Region_0 y Region_2 tienen errores similares y más altos (~40%). Aun y cuando el beneficio es menor es el que tiene menos riesgo de que sea diferente a lo predicho.

In [ ]:
def calcular_beneficio_region(datos, modelo, nombre_region):
    """
    Calcula beneficio y riesgo según las condiciones del proyecto
    """
    # Constantes del proyecto
    PRESUPUESTO_TOTAL = 100_000_000  # 100 millones USD
    POZOS_SELECCIONADOS = 200
    INGRESO_POR_UNIDAD = 4500  # USD por mil barriles
    
    # Preparar datos para predicción
    features = datos.drop(['product','id'], axis=1)
    target = datos['product']
    
 
   # Hacer predicciones en todos los puntos
    predicciones = modelo.predict(features)

    
    # Seleccionar los 200 mejores pozos según predicciones
    indices_mejores = np.argsort(predicciones)[-POZOS_SELECCIONADOS:]
    
  
  # Calcular los ingresos y beneficios de los 200 mejores pozos según predicciones
    volumenes_seleccionados = datos['product'].iloc[indices_mejores]
    volumen_total = volumenes_seleccionados.sum()
    ingresos_totales = volumen_total * INGRESO_POR_UNIDAD
    beneficio_neto = ingresos_totales - PRESUPUESTO_TOTAL

    

    print(f"\n=== ANÁLISIS DE BENEFICIOS - {nombre_region.upper()} ===")
    print(f"Pozos analizados: {len(datos)}")
    print(f"Pozos seleccionados: {POZOS_SELECCIONADOS}")
    print(f"Ingresos totales:  ${ingresos_totales:,.2f}")
    print(f"Beneficio neto: ${beneficio_neto:,.2f}")
    
    
    return indices_mejores, predicciones, beneficio_neto, volumenes_seleccionados

In [ ]:
# Analizar el beneficio de cada región
indices_0, predicciones_0, beneficio_0,volumenes_seleccionados_0 = calcular_beneficio_region(
    region_0, resultados_region_0['modelo'], 'Region_0'
)

indices_1, predicciones_1, beneficio_1,volumenes_seleccionados_1 = calcular_beneficio_region(
    region_1, resultados_region_1['modelo'], 'Region_1'
)

indices_2, predicciones_2, beneficio_2,volumenes_seleccionados_2 = calcular_beneficio_region(
    region_2, resultados_region_2['modelo'], 'Region_2'
)

In [ ]:
def analizar_riesgo_bootstrap(volumenes_seleccionados,beneficio_deterministico, n_simulaciones=1000):
    """
    Calcula el riesgo usando Bootstrap (sin repetir cálculos determinísticos)
    """
    print(f"\n=== SIMULACIONES BOOTSTRAP ({n_simulaciones}) ===")
    print(f"Beneficio determinístico: ${beneficio_deterministico:,.2f}")
    
    # Solo las simulaciones Bootstrap
    beneficios_simulados = []
    for i in range(n_simulaciones):
        
# Muestra aleatoria con reemplazo de 200 pozos
        muestra = np.random.choice(volumenes_seleccionados, size=200, replace=True)

        
        # Calcular beneficio de esta muestra
        ingresos_muestra = muestra.sum() * 4500
        beneficio_muestra = ingresos_muestra - 100000000
        beneficios_simulados.append(beneficio_muestra)
    
    beneficios_simulados = np.array(beneficios_simulados)
    
    # Calcular métricas de riesgo
    
    beneficio_promedio = beneficios_simulados.mean()
    intervalo_confianza = np.percentile(beneficios_simulados, [2.5, 97.5])
    riesgo_perdidas = (beneficios_simulados < 0).mean() * 100

    
    print(f"Beneficio promedio: ${beneficio_promedio:,.2f}")
    print(f"Intervalo de confianza 95%: ${intervalo_confianza[0]:,.2f} - ${intervalo_confianza[1]:,.2f}")
    print(f"Riesgo de pérdidas: {riesgo_perdidas:.1f}%")
    
    return beneficio_deterministico, beneficio_promedio

In [ ]:
resultado = analizar_riesgo_bootstrap(volumenes_seleccionados_0, beneficio_0)

resultado = analizar_riesgo_bootstrap(volumenes_seleccionados_1, beneficio_1)

resultado = analizar_riesgo_bootstrap(volumenes_seleccionados_2, beneficio_2)

Región 0 (Más variable):

Beneficio determinístico: á
34.9M (promedio de 1000 simulaciones)
Intervalo 95%: 
37.7M (rango amplio = más riesgo)
Riesgo de pérdidas: 0% (no se pierde dinero)
Región 1 (Sin variabilidad):

Todos los valores son iguales = Sin incertidumbre
Esto sugiere que los pozos son muy similares entre sí
Región 2 (Moderadamente variable):

Intervalo 95%: 
28.9M (rango medio)
Más predecible que Región 0, pero menos que Región 1
Región 0: Mayor beneficio ó
25.7 pero muy predecible (sin sorpresas) Región 2: Beneficio menor $24.2 con mayor riesgo (resultados más variables 6.6M variabilidad)

In [ ]:
def comparar_regiones_completo():
    """
    Compara las tres regiones con todos los análisis: modelo, beneficios y bootstrap
    """
    print("=" * 80)
    print("                    COMPARACIÓN COMPLETA DE REGIONES")
    print("=" * 80)

# Ejecutar análisis de bootstrap para cada región
    print("\n🔍 EJECUTANDO ANÁLISIS DE BOOTSTRAP...")

    # GEO0
    volumenes_seleccionados_promedio_0,beneficio_promedio_0 = analizar_riesgo_bootstrap(volumenes_seleccionados_0, beneficio_0)

    # GEO1  
    volumenes_seleccionados_promedio_1,beneficio_promedio_1 = analizar_riesgo_bootstrap(volumenes_seleccionados_1, beneficio_1)

    # GEO2
    volumenes_seleccionados_promedio_2,beneficio_promedio_2 = analizar_riesgo_bootstrap(volumenes_seleccionados_2, beneficio_2)
    
    # Crear tabla comparativa
    print("\n" + "=" * 80)
    print("                        TABLA COMPARATIVA")
    print("=" * 80)

 # Encabezados
    print(f"{'MÉTRICA':<25} {'REGION_0':<20} {'REGION_1':<20} {'REGION_2':<15}")
    print("-" * 80)

    # Métricas del modelo
    print(f"{'RMSE':<25} {resultados_region_0['rmse']:<20.2f} {resultados_region_1['rmse']:<20.2f} {resultados_region_2['rmse']:<15.2f}")
    print(f"{'Error relativo (%)':<25} {(resultados_region_0['rmse']/resultados_region_0['volumen_medio_real'])*100:<20.1f} {(resultados_region_1['rmse']/resultados_region_1['volumen_medio_real'])*100:<20.1f} {(resultados_region_2['rmse']/resultados_region_2['volumen_medio_real'])*100:<15.1f}")
    print("-" * 80)

 # Métricas financieras
    print(f"{'Beneficio determinístico':<25} ${beneficio_0/1e6:<19.1f}M ${beneficio_1/1e6:<19.1f}M ${beneficio_2/1e6:<14.1f}M")
    print(f"{'Beneficio medio':<25} ${beneficio_promedio_0/1e6:<19.1f}M ${beneficio_promedio_1/1e6:<19.1f}M ${beneficio_promedio_2/1e6:<14.1f}M")

    print("=" * 80)

    # Agregar estas líneas antes de "# Encontrar la mejor región":
    beneficios = [beneficio_0, beneficio_1, beneficio_2]
    errores = [
    (resultados_region_0['rmse']/resultados_region_0['volumen_medio_real'])*100,
    (resultados_region_1['rmse']/resultados_region_1['volumen_medio_real'])*100,  
    (resultados_region_2['rmse']/resultados_region_2['volumen_medio_real'])*100]

    # Recomendación
    print("\n🎯 RECOMENDACIÓN:")

    # Encontrar la mejor región
    mejor_region = np.argmax(beneficios)
    regiones = ['REGION_0', 'REGION_1', 'REGION_2']
    
    print(f"La mejor región es {regiones[mejor_region]} con:")
    print(f"- Beneficio: ${beneficios[mejor_region]/1e6:.1f}M")
    print(f"- Error del modelo: {errores[mejor_region]:.1f}%")
    
    if errores[mejor_region] > 30:
        print("⚠️  ADVERTENCIA: El modelo tiene un error alto (>30%)")
    
    print(f"\n💡 JUSTIFICACIÓN:")
    print(f"- Mayor beneficio potencial: ${max(beneficios)/1e6:.1f}M")
    print(f"- Riesgo aceptable según análisis Bootstrap")
    print(f"- Todas las regiones tienen 0% riesgo de pérdidas")


In [ ]:
comparar_regiones_completo()